# IGBT Lifetime Prediction
### Junction Temperature → Rainflow Counting → Lifetime Model → Monte Carlo

**Workflow:**
Configure → Connect → Simulate → Rainflow Count → Select Model → Compute Lifetime → (Optional) Monte Carlo → Plot

---

### Pipeline overview

```
PLECS simulation(s) ──► Tj waveform(s) ──► Rainflow counting ──► Cycles (ΔT, Tm, n)
                                                                         │
                                                                Lifetime model (Nf)
                                                                         │
                                                               Miner's rule → Damage
                                                                         │
                                                               Lifetime = T_sim / D
                                                                         │
                                                    (optional) Monte Carlo → distribution
```

When a **sweep** is configured, every simulation step is analysed in parallel:
each step gets its own rainflow histogram and lifetime estimate, and a summary
comparison bar chart is produced automatically.

---

### Lifetime models available

| Key | Model | Formula |
|---|---|---|
| `coffin_manson` | Coffin-Manson | Nf = A · ΔTj⁻ⁿ |
| `modified_coffin_manson` | Modified Coffin-Manson | Nf = A · ΔTj⁻ⁿ · exp(Ea / kB·Tjm) |
| `norris_landzberg` | Norris-Landzberg | adds cycling frequency f |
| `bayerer_2008` | Bayerer (2008) | adds ton, I, V, D, Tjmin |
| `semikron_2013` | Semikron ηρ-model (2013) | adds bond-wire aspect ratio ar, ton |

> **Boltzmann constant** used: $k_B = 1.380\,649 \times 10^{-23}$ J K⁻¹ (SI exact).  
> Activation energies **Ea must be in Joules** — convert from eV: $E_a^{\rm J} = E_a^{\rm eV} \times 1.602\,176\,634 \times 10^{-19}$

---

### Miner's linear damage rule

$$D = \sum_i \frac{n_i}{N_{f,i}} \qquad \text{Lifetime} = \frac{T_{\text{sim}}}{D}$$

---

> Requires `plecs_sim_library.py` **and** `igbt_lifetime_library.py` in the same folder.  
> PLECS Standalone must be running with the XML-RPC interface enabled (default port 1080).

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ─── PLECS interface (existing library — do not modify) ──────────────────────
from src.plecs_sim_library import (
    connect_plecs, load_model, close_model,
    inspect_signals,
    run_sweep, run_montecarlo,
    plot_signals,
)

# ─── IGBT lifetime analysis library ─────────────────────────────────────────
from src.igbt_lifetime_library import *

print("Imports OK.")

Imports OK.


---
## ⚙ Configuration
**This is the only cell you normally need to edit.**

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
#  1. PLECS XML-RPC connection
# ─────────────────────────────────────────────────────────────────────────────
PLECS_HOST = "http://localhost:1080/RPC2"

# ─────────────────────────────────────────────────────────────────────────────
#  2. Model location
# ─────────────────────────────────────────────────────────────────────────────
MODEL_FOLDER = r"\\eistore2\ie-institut\03_Projets\03-01_Recherche\GTHC_SE\2025_137206_P6_ETPS\03-01-02_Technique\03-01-02-02_Simulation\Simulation_PyPLECS"   # <-- edit this
MODEL_NAME   = "20260220_ETPS_DoublePhaseBuck60kHz"                  # <-- WITHOUT .plecs extension

# ─────────────────────────────────────────────────────────────────────────────
#  3. BASE_VARS  (Mode A: populate dict | Mode B: leave empty {})
# ─────────────────────────────────────────────────────────────────────────────
BASE_VARS = {}   # {} = PLECS controls all variables

# ─────────────────────────────────────────────────────────────────────────────
#  4. Sweep parameters
#     SWEEP_PARAMS = {}       → single run
#     SWEEP_PARAMS populated  → parameter sweep (all steps analysed)
# ─────────────────────────────────────────────────────────────────────────────
SWEEP_PARAMS = {
    # Example — sweep thermal models:
    'Therm_mod' : ['file:IMZA120R012M2H-SKG', 'file:IMZA120R017M2H-SKG'],
    # Example — sweep load current:
    # 'I_load': [50, 100, 150, 200],
}   # <-- empty = single run

SWEEP_LABEL_PARAM = 'Therm_mod'  # parameter used for labels; None = auto (first key)

# ─────────────────────────────────────────────────────────────────────────────
#  5. Execution mode
#  #     'simulate' -> run time-domain simulation
#  #     'analyze'  -> run steady-state or other analysis
# ─────────────────────────────────────────────────────────────────────────────
EXEC_MODE = 'analyze'   # <-- 'simulate' or 'analyze'

# ─────────────────────────────────────────────────────────────────────────────
#  6. Analysis name  (used when EXEC_MODE = 'analyze')
# ─────────────────────────────────────────────────────────────────────────────
ANALYSIS_NAME = 'Steady-State Analysis'   # <-- name of the analysis in PLECS

# ─────────────────────────────────────────────────────────────────────────────
#  7. Signal names  (in Outport wiring order, top to bottom)
#     Use inspect_signals(results) in Step 4 if you are unsure of the order.
# ─────────────────────────────────────────────────────────────────────────────
SIGNAL_NAMES = [
    'Junction temperature IGBT',   # signal_index = 0   <-- edit
    # 'Junction temperature Diode',  # signal_index = 1
]

TJ_SIGNAL_INDEX = 0   # signal_index of the Tj trace used for rainflow

# ─────────────────────────────────────────────────────────────────────────────
#  8. Mission profile duration
#     Real-world time represented by ONE simulation run.
#       10-minute profile  → T_SIM_HOURS = 10 / 60
#       Full year          → T_SIM_HOURS = 8760
# ─────────────────────────────────────────────────────────────────────────────
T_SIM_HOURS = 0.3/60   # [h]

# ─────────────────────────────────────────────────────────────────────────────
#  9. Rainflow histogram bins
# ─────────────────────────────────────────────────────────────────────────────
N_DELTA_T_BINS = 8
N_TMEAN_BINS   = 8

# ─────────────────────────────────────────────────────────────────────────────
#  10. Lifetime model
#     Run print_model_table() in Step 7 to read model descriptions.
#     Options: 'coffin_manson' | 'modified_coffin_manson' |
#              'norris_landzberg' | 'bayerer_2008' | 'semikron_2013'
# ─────────────────────────────────────────────────────────────────────────────
MODEL_KEY = 'modified_coffin_manson'   # <-- choose your model

# ─────────────────────────────────────────────────────────────────────────────
#  11. Model parameters
#     Run print_model_parameters(MODEL_KEY) in Step 7 to see all defaults.
#
#     kB = 1.380649e-23 J/K  →  Ea must be in Joules!
#     Conversion: Ea_J = Ea_eV × 1.602176634e-19
#       0.63 eV  →  1.009e-19 J
#       0.06 eV  →  9.613e-21 J
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PARAMS = {
    # ── Coffin-Manson / Modified CM / Norris-Landzberg ────────────────────────
    'A'  : 3.3e15,       # Pre-exponential constant
    'n'  : 4.0,           # Coffin-Manson exponent
    'Ea' : 1.009e-19,     # Activation energy [J]  
    # ── Norris-Landzberg only ─────────────────────────────────────────────────
    # 'f'  : 0.02,        # Cycling frequency [Hz]
    # 'm'  : 0.5,         # Frequency exponent
    # ── Bayerer 2008 only ─────────────────────────────────────────────────────
    # 'beta1': -4.416, 'beta2': 1285.0, 'beta3': -0.463,
    # 'beta4': -0.716, 'beta5': -0.761, 'beta6': -0.5,
    # 'ton': 1.0, 'I': 100.0, 'V': 600.0, 'D': 375e-6,
    # ── Semikron 2013 only ────────────────────────────────────────────────────
    # 'beta0': -4.923, 'beta1': -9.012, 'C': 1.434,
    # 'gamma': 0.5, 'ar': 3.1e-3, 'ton': 1.0, 'fdiode': 1.0,
    # 'Ea': 9.613e-21,    # 0.06 eV for Semikron bond-wire mechanism
}

# ─────────────────────────────────────────────────────────────────────────────
#  12. Monte Carlo — uncertainty on model parameters
#      Format: { 'param_name' : (relative_tolerance, distribution) }
#        'uniform' → flat ± tol  |  'normal' → Gaussian σ = nominal × tol
#      MC_RESULT_INDEX selects which sweep result to analyse with MC.
# ─────────────────────────────────────────────────────────────────────────────
ENABLE_MC       = True
N_MC_RUNS       = 300
MC_SEED         = 42
MC_RESULT_INDEX = 0   # which sweep result to run Monte Carlo on

MC_CONFIG = {
    'A'  : (0.15, 'uniform'),   # ±15 % flat
    'n'  : (0.05, 'normal'),    #  5 % sigma
    'Ea' : (0.10, 'normal'),    # 10 % sigma
}

# ─────────────────────────────────────────────────────────────────────────────
#  13. Tj time-domain plot configuration
# ─────────────────────────────────────────────────────────────────────────────
PLOT_CONFIG = [
    {
        'signal_index' : TJ_SIGNAL_INDEX,
        'signal_name'  : SIGNAL_NAMES[TJ_SIGNAL_INDEX],
        'time_window'  : None,
        'title'        : 'Junction Temperature — PLECS simulation',
        'xlabel'       : 'Time (s)',
        'ylabel'       : 'Temperature (°C)',
        'show_legend'  : True,
    },
]
MAX_POINTS = 50_000

# ─────────────────────────────────────────────────────────────────────────────
#  14. Annual operating hours (for lifetime unit conversion)
# ─────────────────────────────────────────────────────────────────────────────
OPERATING_HOURS_PER_YEAR = 8760.0

# ─── Summary ─────────────────────────────────────────────────────────────────
print("Configuration loaded.")
print(f"  Model              : {MODEL_KEY}")
print(f"  Tj signal index    : {TJ_SIGNAL_INDEX}  ({SIGNAL_NAMES[TJ_SIGNAL_INDEX]})")
print(f"  Simulation duration: {T_SIM_HOURS} h")
print(f"  Rainflow bins      : ΔT={N_DELTA_T_BINS}  ×  T_mean={N_TMEAN_BINS}")
print(f"  Sweep params       : {list(SWEEP_PARAMS.keys()) or 'none (single run)'}")
print(f"  Monte Carlo        : {'enabled — ' + str(N_MC_RUNS) + ' runs' if ENABLE_MC else 'disabled'}")

Configuration loaded.
  Model              : modified_coffin_manson
  Tj signal index    : 0  (Junction temperature IGBT)
  Simulation duration: 0.005 h
  Rainflow bins      : ΔT=8  ×  T_mean=8
  Sweep params       : ['Therm_mod']
  Monte Carlo        : enabled — 300 runs


---
## Step 1 — Connect to PLECS

In [4]:
server = connect_plecs(PLECS_HOST)

[PLECS] Connected to http://localhost:1080/RPC2


---
## Step 2 — Load the PLECS Model

In [5]:
load_model(server, MODEL_FOLDER, MODEL_NAME)

[PLECS] Model '20260220_ETPS_DoublePhaseBuck60kHz' loaded from '\\eistore2\ie-institut\03_Projets\03-01_Recherche\GTHC_SE\2025_137206_P6_ETPS\03-01-02_Technique\03-01-02-02_Simulation\Simulation_PyPLECS'


---
## Step 3 — Run PLECS Simulation(s)

- `SWEEP_PARAMS = {}` → one simulation, wrapped in a single-element list.
- `SWEEP_PARAMS` populated → full parameter sweep; **every step** will be
  analysed independently in Steps 5–8.

In [6]:
if SWEEP_PARAMS:
    results = run_sweep(
        server, MODEL_NAME, BASE_VARS, SWEEP_PARAMS,
        mode=EXEC_MODE, analysis_name=ANALYSIS_NAME if EXEC_MODE == 'analyze' else None
    )
else:
    # Single run — use a one-step no-op sweep so the output is always a list
    results = run_sweep(
        server, MODEL_NAME, BASE_VARS,
        sweep_params={'_dummy': [0]},
        mode=EXEC_MODE, analysis_name=ANALYSIS_NAME if EXEC_MODE == 'analyze' else None
    )
    results[0]['params'] = {}

N_RESULTS = len(results)
print(f"\n{N_RESULTS} simulation result(s) available.")

[Sweep] 2 step(s) | swept: ['Therm_mod'] | mode: PLECS vars only (base_vars empty)
  step   1/2  ->  {'Therm_mod': 'file:IMZA120R012M2H-SKG'}
  step   2/2  ->  {'Therm_mod': 'file:IMZA120R017M2H-SKG'}
[Sweep] Done.

2 simulation result(s) available.


---
## Step 4 — Inspect Signals & Plot Junction Temperature

All sweep steps are overlaid on a single Plotly figure.  
Use `inspect_signals(results)` to confirm which row index maps to Tj,  
then update `TJ_SIGNAL_INDEX` in the Configuration cell.

In [7]:
inspect_signals(results)

Signals available in results (first run):
  Total signals  : 2  -> use signal_index = 0 to 1
  Total samples  : 3244740
  Time range     : 0 s  to  0.9 s

  index                min             max            mean
  -------    -------------   -------------   -------------
  0                 108.01          114.31          110.68
  1                 100.37          112.71          104.82


In [8]:
# All sweep steps are plotted on the same interactive figure
plot_signals(
    results,
    PLOT_CONFIG,
    sim_mode    = 'sweep' if SWEEP_PARAMS else 'single',
    label_param = SWEEP_LABEL_PARAM,
    max_points  = MAX_POINTS,
)

[Plot] 'Junction Temperature — PLECS simulation'
       2 trace(s) | 6,489,480 raw pts -> 64,480 rendered (101x reduction) | build 0.09s | render 1.97s


---
## Step 5 — Rainflow Cycle Counting  *(all simulation results)*

The ASTM E1049-85 three-point algorithm runs on every sweep step and the
results are stored in `cycles_list[i]`.

| Column | Meaning | Unit |
|---|---|---|
| `range` | ΔTj — temperature swing | °C |
| `mean`  | Tjm — mean junction temperature | °C |
| `count` | n   — 0.5 (half-cycle) or 1.0 (full cycle) | – |

In [9]:
cycles_list = []   # one DataFrame per simulation result

header = (f"{'Step':<6}  {'Parameters':<42}  {'Cycles':>8}  "
          f"{'Total count':>12}  {'ΔT min':>8}  {'ΔT max':>8}")
print(header)
print('─' * len(header))

for idx, res in enumerate(results):
    tj  = res['values'][TJ_SIGNAL_INDEX]
    cyc = rainflow_counting(tj)
    cycles_list.append(cyc)

    params_str = str(res['params']) if res['params'] else '(single run)'
    if len(params_str) > 40:
        params_str = params_str[:37] + '...'

    if cyc.empty:
        print(f"{idx:<6}  {params_str:<42}  {'—':>8}  {'—':>12}  {'—':>8}  {'—':>8}")
    else:
        print(f"{idx:<6}  {params_str:<42}  "
              f"{len(cyc):>8}  {cyc['count'].sum():>12.1f}  "
              f"{cyc['range'].min():>8.2f}  {cyc['range'].max():>8.2f}")

print()
print(f"Stored {len(cycles_list)} rainflow result(s) in cycles_list[0..{len(cycles_list)-1}]")
print("\nPreview — cycles_list[0]:")
cycles_list[0].head(10)

Step    Parameters                                    Cycles   Total count    ΔT min    ΔT max
──────────────────────────────────────────────────────────────────────────────────────────────
0       {'Therm_mod': 'file:IMZA120R012M2H-SKG'}      107806      107800.5      0.00      6.29
1       {'Therm_mod': 'file:IMZA120R017M2H-SKG'}      107750      107745.5      0.00      4.02

Stored 2 rainflow result(s) in cycles_list[0..1]

Preview — cycles_list[0]:


,range,mean,count
0,0.094109,109.218444,0.5
1,0.041917,109.359318,1.0
2,0.226738,109.284758,0.5
3,0.041923,109.359070,1.0
4,0.226826,109.284463,1.0
5,0.042044,109.358944,1.0
6,0.226909,109.284235,1.0
7,0.042183,109.359181,1.0
8,0.227275,109.284217,1.0
9,0.042313,109.359880,1.0


---
## Step 6 — 3-D Cycle Histograms  *(one interactive chart per result)*

**X** = mean junction temperature $T_m$ (°C)  
**Y** = temperature swing $\Delta T$ (°C)  
**Z** = cycle count $N$ in that bin  

Each chart is interactive: drag to rotate, scroll to zoom, hover for exact values.

In [10]:
figs_3d = []   # store Figure objects (useful for export)

for idx, cyc in enumerate(cycles_list):
    if cyc.empty:
        print(f"[Step {idx}] No cycles — skipping histogram.")
        figs_3d.append(None)
        continue

    # Build a short label from the swept parameters
    params = results[idx]['params']
    if params:
        lp  = SWEEP_LABEL_PARAM or list(params.keys())[0]
        v   = params[lp]
        lbl = f"[{lp}={v:.4g}]" if isinstance(v, float) else f"[{v}]"
    else:
        lbl = ''

    fig = plot_3d_cycle_histogram(
        cyc,
        n_delta_T_bins = N_DELTA_T_BINS,
        n_T_mean_bins  = N_TMEAN_BINS,
        title          = 'Rainflow Cycle Count — Junction Temperature',
        label          = lbl,
        colormap       = 'viridis',   # try 'plasma', 'Blues', 'YlOrRd', 'coolwarm'
        opacity        = 0.88,
        bar_gap        = 0.12,
    )
    figs_3d.append(fig)

[3D Histogram] 'Rainflow Cycle Count — Junction Temperature  [file:IMZA120R012M2H-SKG]'
  Total cycles  : 107800.5  |  Non-empty bins : 13 / 64
  ΔT range      : [0.00, 6.29] °C
  T_m range     : [108.02,  114.17] °C


[3D Histogram] 'Rainflow Cycle Count — Junction Temperature  [file:IMZA120R017M2H-SKG]'
  Total cycles  : 107745.5  |  Non-empty bins : 11 / 64
  ΔT range      : [0.00, 4.02] °C
  T_m range     : [83.75,  87.63] °C


---
## Step 7 — Lifetime Model Selection

Run the two cells below to read the model descriptions and their default
parameters, then update `MODEL_KEY` and `MODEL_PARAMS` in the **Configuration**
cell and re-run from there.

In [11]:
print_model_table()


════════════════════════════════════════════════════════════════════════
  IGBT LIFETIME MODELS  —  Overview
════════════════════════════════════════════════════════════════════════

  [coffin_manson]
  Name    : Coffin-Manson  (classic)
  Formula : Nf = A · ΔTj^(-n)
  Inputs  : delta_T
            The original Coffin-Manson model relates the number of cycles to
            failure solely to the junction temperature swing ΔTj. It is the
            simplest model and serves as a baseline. It does NOT account for
            mean temperature effects, making it suitable only when temperature
            swings are the dominant ageing driver and operating conditions are
            roughly constant.

  Parameters:
    A          default=344000000000000.0  Pre-exponential constant  (fit to device data)  [–]
    n          default=5.0           Coffin-Manson exponent  (typically 4–7 for IGBTs)  [–]

  [modified_coffin_manson]
  Name    : Modified Coffin-Manson  (classic+Ea)
  Formula : Nf 

In [12]:
default_params = print_model_parameters(MODEL_KEY)


────────────────────────────────────────────────────────────
  Model : Modified Coffin-Manson  (classic+Ea)
  Nf = A · ΔTj^(-n) · exp(Ea / (kB · T_jm))
────────────────────────────────────────────────────────────
  Parameter            Default  Unit    Description
  ─────────         ──────────  ────    ────────────────────────────
  A                   3.44e+14  –       Pre-exponential constant
  n                          5  –       Coffin-Manson exponent
  Ea                 1.009e-19  J       Activation energy  (0.4–0.9 eV → 6.4e-20–1.44e-19 J)
────────────────────────────────────────────────────────────



---
## Step 8 — Deterministic Lifetime  *(all simulation results)*

Miner's rule is applied to **every** sweep step using the same model and
parameters. Results are printed, collected in `df_lifetime`, and plotted as
a comparison bar chart.

$$D = \sum_i \frac{n_i}{N_{f,i}} \qquad \text{Lifetime} = \frac{T_{\text{sim}}}{D}$$

In [13]:
lifetime_records = []

print(f"Model : {LIFETIME_MODELS[MODEL_KEY]['name']}  |  T_sim = {T_SIM_HOURS} h")
print('─' * 82)

for idx, cyc in enumerate(cycles_list):
    params = results[idx]['params']

    if cyc.empty:
        print(f"  Step {idx}: no cycles — skipped.")
        continue

    Nf     = compute_nf_per_cycle(cyc, MODEL_KEY, MODEL_PARAMS)
    damage = miner_damage(cyc, Nf)
    lt     = lifetime_from_damage(damage, T_SIM_HOURS, OPERATING_HOURS_PER_YEAR)

    row = {
        'step':           idx,
        'damage':         damage,
        'lifetime_h':     lt['hours'],
        'lifetime_days':  lt['days'],
        'lifetime_years': lt['years'],
    }
    row.update(params)   # add swept parameter values as extra columns
    lifetime_records.append(row)

    # Build display label
    if params:
        lp  = SWEEP_LABEL_PARAM or list(params.keys())[0]
        lbl = f"{lp}={params[lp]}"
    else:
        lbl = 'single run'

    days_str  = f"{lt['days']:.3e}" if lt['days'] > 1e6 else f"{lt['days']:.2f}"
    years_str = f"{lt['years']:.3e}" if lt['years'] > 1e4 else f"{lt['years']:.3f}"
    print(f"  Step {idx} [{lbl}]:  damage = {damage:.3e}  "
          f"→  {days_str} days  ({years_str} years)")

df_lifetime = pd.DataFrame(lifetime_records)
print('─' * 82)
print("\nLifetime summary DataFrame:")
df_lifetime

Model : Modified Coffin-Manson  |  T_sim = 0.005 h
──────────────────────────────────────────────────────────────────────────────────
  Step 0 [Therm_mod=file:IMZA120R012M2H-SKG]:  damage = 1.282e-20  →  1.625e+16 days  (4.452e+13 years)
  Step 1 [Therm_mod=file:IMZA120R017M2H-SKG]:  damage = 7.097e-22  →  2.936e+17 days  (8.043e+14 years)
──────────────────────────────────────────────────────────────────────────────────

Lifetime summary DataFrame:


,step,damage,lifetime_h,lifetime_days,lifetime_years,Therm_mod
0,0,1.281949e-20,3.900312e+17,1.625130e+16,4.452411e+13,file:IMZA120R012M2H-SKG
1,1,7.096920e-22,7.045310e+18,2.935546e+17,8.042591e+14,file:IMZA120R017M2H-SKG


In [14]:
if not df_lifetime.empty:

    # ── Build x-axis labels from swept parameters ──────────────────────────────
    def _step_label(row):
        reserved = {'step','damage','lifetime_h','lifetime_days','lifetime_years'}
        extras   = [k for k in row.index if k not in reserved]
        if not extras:
            return f"Step {int(row['step'])}"
        lp = SWEEP_LABEL_PARAM or extras[0]
        v  = row[lp]
        return f"{lp}={v:.4g}" if isinstance(v, float) else f"{lp}={v}"

    x_labels = [_step_label(r) for _, r in df_lifetime.iterrows()]
    y_days   = df_lifetime['lifetime_days'].tolist()

    # Cap infinite/very large values for display; annotate with actual text
    y_cap    = [min(v, 1e9) if np.isfinite(v) else 1e9 for v in y_days]
    y_text   = [f"{v:.2e} d" if v > 1e6 else f"{v:.1f} d" for v in y_days]

    use_log  = (
        len(y_cap) > 1
        and max(y_cap) / (min(v for v in y_cap if v > 0) + 1e-9) > 50
    )

    palette  = px.colors.qualitative.Plotly if len(x_labels) <= 10 else None
    colors   = (palette[:len(x_labels)] if palette
                else ['steelblue'] * len(x_labels))

    fig_cmp = go.Figure(go.Bar(
        x             = x_labels,
        y             = y_cap,
        marker_color  = colors,
        text          = y_text,
        textposition  = 'outside',
        hovertemplate = '<b>%{x}</b><br>Lifetime: %{text}<extra></extra>',
    ))
    fig_cmp.update_layout(
        title       = (f'Deterministic Lifetime Comparison — '
                       f'{LIFETIME_MODELS[MODEL_KEY]["name"]}'),
        xaxis_title = 'Simulation step',
        yaxis_title = 'Lifetime (days)',
        yaxis_type  = 'log' if use_log else 'linear',
        template    = 'plotly_white',
    )
    fig_cmp.show()
else:
    print("No lifetime data to plot.")

In [15]:
# ── Per-cycle damage breakdown for a selected result ──────────────────────────
DETAIL_INDEX = 0   # <-- which result to inspect

cyc_d  = cycles_list[DETAIL_INDEX].copy()
Nf_d   = compute_nf_per_cycle(cyc_d, MODEL_KEY, MODEL_PARAMS)
cyc_d['Nf']     = Nf_d
cyc_d['damage'] = cyc_d['count'] / Nf_d
cyc_d = cyc_d.sort_values('damage', ascending=False)

print(f"Top 10 most damaging cycles — result {DETAIL_INDEX}:")
print(cyc_d.head(10).to_string(index=False, float_format=lambda x: f"{x:.4g}"))

Top 10 most damaging cycles — result 0:
 range  mean  count        Nf    damage
 6.293 111.2      1 3.818e+20 2.619e-21
 6.293 111.2      1 3.818e+20 2.619e-21
 6.293 111.2      1 3.819e+20 2.619e-21
 6.293 111.2    0.5 3.818e+20 1.309e-21
 6.293 111.2    0.5 3.818e+20 1.309e-21
 6.293 111.2    0.5 3.818e+20 1.309e-21
 5.135 111.7    0.5 8.366e+20 5.976e-22
 2.612 109.3    0.5 1.409e+22 3.547e-23
 1.455 109.9      1 1.423e+23 7.025e-24
 1.455 109.9      1 1.423e+23 7.025e-24


---
## Step 9 — Monte Carlo Lifetime Analysis *(optional)*

Model parameters carry uncertainty from curve-fitting. Monte Carlo propagates
that uncertainty to a **lifetime distribution**.

The MC runs on the result selected by `MC_RESULT_INDEX` (default 0).  
Set `ENABLE_MC = False` in Configuration to skip.

| Distribution | Sampling rule |
|---|---|
| `'uniform'` | drawn from [nominal × (1−tol), nominal × (1+tol)] |
| `'normal'`  | Gaussian with σ = \|nominal\| × tol |

In [16]:
mc_results = None

if ENABLE_MC:
    mc_idx = min(MC_RESULT_INDEX, len(cycles_list) - 1)
    if mc_idx != MC_RESULT_INDEX:
        print(f"MC_RESULT_INDEX adjusted from {MC_RESULT_INDEX} to {mc_idx} "
              f"(only {len(cycles_list)} result(s) available).")

    mc_label = results[mc_idx]['params'] or '(single run)'
    print(f"Running Monte Carlo on result {mc_idx}: {mc_label}\n")

    mc_results = run_montecarlo_lifetime(
        cycles                   = cycles_list[mc_idx],
        model_key                = MODEL_KEY,
        nominal_params           = MODEL_PARAMS,
        mc_config                = MC_CONFIG,
        T_sim_hours              = T_SIM_HOURS,
        n_runs                   = N_MC_RUNS,
        seed                     = MC_SEED,
        operating_hours_per_year = OPERATING_HOURS_PER_YEAR,
    )
else:
    print("Monte Carlo skipped (ENABLE_MC = False).")

Running Monte Carlo on result 0: {'Therm_mod': 'file:IMZA120R012M2H-SKG'}

[MC Lifetime] model='modified_coffin_manson'  n_runs=300  seed=42
  Perturbed parameters: ['A', 'n', 'Ea']
  Mission profile: 0.005 h
  [  30/300]  lifetime = 32124463698377908.00 days
  [  60/300]  lifetime = 32577315343128596.00 days
  [  90/300]  lifetime = 31683164469077024.00 days
  [ 120/300]  lifetime = 18269093566115664.00 days
  [ 150/300]  lifetime = 180311310161636640.00 days
  [ 180/300]  lifetime = 3046849634818352.50 days
  [ 210/300]  lifetime = 21475567829370636.00 days
  [ 240/300]  lifetime = 1695686694202875.00 days
  [ 270/300]  lifetime = 24466147023543548.00 days
  [ 300/300]  lifetime = 46343711348341544.00 days
[MC Lifetime] Done. 300 successful runs.


In [17]:
if mc_results is not None and not mc_results.empty:
    print("Monte Carlo summary statistics:")
    print(mc_results[['damage', 'lifetime_h', 'lifetime_days', 'lifetime_years']]
          .describe()
          .to_string(float_format=lambda x: f"{x:.4g}"))

Monte Carlo summary statistics:
         damage  lifetime_h  lifetime_days  lifetime_years
count       300         300            300             300
mean  7.571e-20    2.05e+18      8.541e+16        2.34e+14
std   2.066e-19   9.116e+18      3.798e+17       1.041e+15
min   3.565e-23   2.371e+15      9.879e+13       2.707e+11
25%    4.55e-21   9.219e+16      3.841e+15       1.052e+13
50%   1.181e-20   4.233e+17      1.764e+16       4.832e+13
75%   5.424e-20   1.099e+18      4.579e+16       1.254e+14
max   2.109e-18   1.403e+20      5.844e+18       1.601e+16


---
## Step 10 — Monte Carlo Lifetime Distribution Plot

Interactive histogram with fitted normal PDF and P5 / P50 / P95 percentile lines.

Change the `column` argument to select the time unit:
- `'lifetime_days'`
- `'lifetime_years'`
- `'lifetime_h'`

In [18]:
if mc_results is not None and not mc_results.empty:
    fig_mc = plot_lifetime_histogram(
        mc_results,
        column      = 'lifetime_days',
        n_bins      = 30,
        fit_normal  = True,
        title       = (f'MC Lifetime Distribution — {LIFETIME_MODELS[MODEL_KEY]["name"]} '
                       f'(result {MC_RESULT_INDEX})'),
        xlabel      = 'Lifetime (days)',
        percentiles = [5, 50, 95],
        color       = 'steelblue',
    )
else:
    print("No Monte Carlo results to plot.")


[MC Lifetime Histogram]  column='lifetime_days'  n=300
  Mean   : 8.541e+16
  Std    : 3.792e+17  (444.0% of mean)
  Median : 1.764e+16
  P5     : 6.103e+14
  P50    : 1.764e+16
  P95    : 2.95e+17


---
## Step 11 — Close the PLECS Model

In [19]:
close_model(server, MODEL_NAME)

[PLECS] Model '20260220_ETPS_DoublePhaseBuck60kHz' closed.
